In [ ]:
import os
import numpy as np
import keras
from keras import layers, Sequential
import tensorflow as tf
from tensorflow import data as tf_data
import matplotlib.pyplot as plt
from keras.layers import Rescaling, RandomFlip, RandomRotation, RandomZoom
import pickle
from pathlib import Path

In [100]:
SEED = 42
IMAGE_SIZE = (128, 128)
INPUT_SIZE = (128, 128, 3)
BATCH_SIZE = 16

# SCRIPT_DIR = Path(__file__).resolve().parent
# PROJECT_ROOT = SCRIPT_DIR.parent
PROJECT_ROOT = Path("../")
DATA_DIR = PROJECT_ROOT / "data/processed"
MODELS_DIR = PROJECT_ROOT / "models"

In [117]:
test_ds = keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'test'),
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    pad_to_aspect_ratio=True,
    shuffle=False,      # <-- add this
    seed=SEED
)
test_ds = test_ds.map(
    lambda img, label: (img / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.cache()          # optional but good: locks in the order/data
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

Found 564 files belonging to 4 classes.


In [ ]:
with open("../models/ResNet_history.pkl", "rb") as file:
    loaded_history = pickle.load(file)

In [82]:
loaded_history.model

<Functional name=ResNet, built=True>

In [83]:
loaded_history.history

{'accuracy': [0.8807726502418518,
  0.960035502910614,
  0.9698046445846558,
  0.972468912601471,
  0.9755772352218628],
 'loss': [0.4242062568664551,
  0.15855123102664948,
  0.11492681503295898,
  0.09379114210605621,
  0.07983177900314331],
 'val_accuracy': [0.9520426392555237,
  0.9609236121177673,
  0.9591474533081055,
  0.9591474533081055,
  0.9591474533081055],
 'val_loss': [0.21873660385608673,
  0.1599547415971756,
  0.13860762119293213,
  0.12761957943439484,
  0.11911013722419739],
 'epoch_time': [41.30222010612488,
  36.73589110374451,
  36.26760506629944,
  34.85313892364502,
  34.91873288154602]}

In [ ]:
# Training Loss
# Validation Loss
# Training Accuracy
# Validation Accuracy
# Training Time
# Test Accuracy

# Precision
# Recall
# F1-score
# Confusion Matrix

In [84]:
training_loss = loaded_history.history['loss']
validation_loss = loaded_history.history['val_loss']
training_accuracy = loaded_history.history['accuracy']
validation_accuracy = loaded_history.history['val_accuracy']
training_time = loaded_history.history['epoch_time']

training_loss, validation_loss, training_accuracy, validation_accuracy, training_time

([0.4242062568664551,
  0.15855123102664948,
  0.11492681503295898,
  0.09379114210605621,
  0.07983177900314331],
 [0.21873660385608673,
  0.1599547415971756,
  0.13860762119293213,
  0.12761957943439484,
  0.11911013722419739],
 [0.8807726502418518,
  0.960035502910614,
  0.9698046445846558,
  0.972468912601471,
  0.9755772352218628],
 [0.9520426392555237,
  0.9609236121177673,
  0.9591474533081055,
  0.9591474533081055,
  0.9591474533081055],
 [41.30222010612488,
  36.73589110374451,
  36.26760506629944,
  34.85313892364502,
  34.91873288154602])

In [86]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [108]:
test_loss, test_accuracy = loaded_history.model.evaluate(test_ds, batch_size=32, verbose=1)
test_accuracy

36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 114ms/step - accuracy: 0.9628 - loss: 0.0984


0.9627659320831299

In [ ]:
def testing(model):
    y_true = np.concatenate([y for x, y in test_ds], axis=0)   # iteration #1 → one shuffle order
    y_pred_probs = model.predict(test_ds)                        # iteration #2 → a DIFFERENT shuffle order

    # For MULTI-CLASS classification (softmax output), uncomment line below:
    y_pred = np.argmax(y_pred_probs, axis=1)

    # --- SCIKIT-LEARN METRICS ---

    print("=== Performance Metrics ===")
    # 'macro' calculates metrics for each class independently, then takes the average
    # 'weighted' accounts for class imbalance by weighting by the number of true instances
    print(f"Precision: {precision_score(y_true, y_pred, average='macro'):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, average='macro'):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred, average='macro'):.4f}\n")


    print("=== Confusion Matrix ===")
    print(confusion_matrix(y_true, y_pred))
    print()

    print("=== Classification Report ===")
    print(classification_report(y_true, y_pred))



In [118]:
testing(loaded_history.model)

36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 122ms/step
=== Performance Metrics ===
Precision: 0.9610
Recall:    0.9634
F1 Score:  0.9619

=== Confusion Matrix ===
[[139   8   1   2]
 [  2 111   1   0]
 [  0   0 149   1]
 [  2   1   3 144]]

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.97      0.93      0.95       150
           1       0.93      0.97      0.95       114
           2       0.97      0.99      0.98       150
           3       0.98      0.96      0.97       150

    accuracy                           0.96       564
   macro avg       0.96      0.96      0.96       564
weighted avg       0.96      0.96      0.96       564

